# Buổi 3 — Bài tập về nhà

Nộp bài qua Hệ thống học tập, trao đổi thêm ở Facebook Group lớp học.

Hai phần: hoàn thiện Bronze và Silver cho hai bảng còn lại (`order_items`, `products`), rồi tạo bảng Gold `daily_revenue`. Cần có bảng `stylehub_silver.orders` từ Bài tập 3.

**Lưu ý nguồn Raw:** cả hai bảng lấy từ Raw Zone đã ghi trong bucket ở Buổi 2, không cần kéo lại từ nguồn.
- `order_items` là bài về nhà Buổi 2. Chưa làm xong thì dùng bucket dự phòng GV gửi trong Facebook Group lớp học, đổi `DEST_BUCKET` trong `.env`.
- `products` ghi đè theo ngày ở Raw Zone (mỗi ngày một snapshot, Bài tập 2 Buổi 2). Ở Bronze thì giữ mọi snapshot, nên cùng một sản phẩm xuất hiện nhiều lần. Silver chọn snapshot mới nhất.

### Cấu hình

Dùng lại đúng khối cấu hình của Buổi 2 (`c2-practice_01`): cùng tên biến, cùng file `.env`, cùng cách tạo client. Trước hết import các thư viện dùng cho cả bài.

In [8]:
import glob
import os
import time
import uuid
from datetime import date, datetime, timezone, timedelta

import pandas as pd
import sqlalchemy as sa
from dotenv import load_dotenv
from google.cloud import storage
from google.oauth2 import service_account

load_dotenv("../../secrets/.env",override=True)

True

Cấu hình đọc từ biến môi trường (file `.env`), nên mật khẩu và tên project không nằm trong code. File `.env` và file key service account đã chuẩn bị từ Buổi 2. `LOCAL_PATH` là folder local giữ bản sao của Raw Zone, dùng chung cho mọi buổi: mỗi file ghi vào đây trước, rồi mới đẩy lên bucket dưới `raw/stylehub/`.

In [ ]:
GCP_PROJECT = os.environ.get("GCP_PROJECT")
DEST_BUCKET = os.environ.get("DEST_BUCKET")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "asia-southeast1")
GCP_KEY_PATH = os.environ.get("GCP_KEY_PATH", "")
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT", 6543)
DB_USER = os.environ.get("DB_USER")
DB_PWD = os.environ.get("DB_PWD")
DB_NAME = os.environ.get("DB_NAME", "postgres")

LOCAL_PATH = "../../output/raw/stylehub"

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PWD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

Tạo client GCS bằng key service account và kiểm tra bucket cá nhân (tạo ở Buổi 1). Bucket đã có thì dùng lại, chưa có thì tạo mới.

In [10]:
credentials = service_account.Credentials.from_service_account_file(GCP_KEY_PATH)
gcs_client = storage.Client(project=GCP_PROJECT, credentials=credentials)

bucket = gcs_client.bucket(DEST_BUCKET)
if not bucket.exists():
    bucket = gcs_client.create_bucket(DEST_BUCKET, location=GCP_LOCATION)
    print(f"Created bucket: gs://{DEST_BUCKET}")
else:
    print(f"Bucket exists: gs://{DEST_BUCKET}")

Bucket exists: gs://de01_laurenprj123


Buổi 3 làm việc với BigQuery nên thêm client BigQuery. Dataset phải cùng location với bucket thì mới nạp được file từ GCS.

In [11]:
from google.cloud import bigquery

bq = bigquery.Client(project=GCP_PROJECT, credentials=credentials, location=GCP_LOCATION)

### Phần 1a: order_items

Nạp Raw vào Bronze, chỉ thêm. Làm giống phần orders ở Bài tập 3, đổi đường dẫn Raw và tên bảng.

In [13]:
files = glob.glob(f"{LOCAL_PATH}/order_items/**/*.parquet", recursive=True)

if not files:
    for blob in bucket.list_blobs(prefix="raw/stylehub/order_items/"):
        local_file = f"{LOCAL_PATH}/{blob.name.removeprefix('raw/stylehub/')}"
        os.makedirs(os.path.dirname(local_file), exist_ok=True)
        blob.download_to_filename(local_file)
    files = glob.glob(f"{LOCAL_PATH}/order_items/**/*.parquet", recursive=True)

if files:
    print(len(files), "file order_items found in local or GCS.")
else:
    print("Không có data order_items ở local lẫn GCS. Chạy phần 'Extract full' ngay bên dưới, rồi chạy lại cell này.")

15 file order_items found in local or GCS.


In [14]:
for f in glob.glob(f"{LOCAL_PATH}/order_items/**/*.parquet", recursive=True):
    name = "raw/stylehub/" + os.path.relpath(f, LOCAL_PATH).replace("\\", "/")
    blob = bucket.blob(name)
    if not blob.exists():
        blob.upload_from_filename(f)

In [18]:
job_config = bigquery.LoadJobConfig(source_format="PARQUET", write_disposition="WRITE_APPEND")
uris = [f"gs://{DEST_BUCKET}/{b.name}"
          for b in bucket.list_blobs(prefix="raw/stylehub/order_items/")
          if b.name.endswith(".parquet") and b.size > 10_000]  # file rỗng chỉ ~5 KB
bq.load_table_from_uri(uris, f"{GCP_PROJECT}.stylehub_bronze.order_items",job_config=job_config).result()

LoadJob<project=laurenprj123, location=asia-southeast1, id=19885302-9912-4333-bff0-6bdab625618a>

In [ ]:
# TODO: nạp order_items từ Raw vào Bronze, làm giống Phần A
# Đường dẫn Raw: raw/stylehub/order_items/date=<data_date>/batch_id=<batch_id>/order_items.parquet
# Nạp mọi file của bảng bằng một dấu * ở cuối: raw/stylehub/order_items/*

Kiểm tra Bronze: số dòng so với số id.

In [47]:
bq.query(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT id) AS n_ids FROM `{GCP_PROJECT}.stylehub_bronze.order_items`").to_dataframe()

,n_rows,n_ids
0,366520,183260


Bronze → Silver: mỗi dòng order_items một bản mới nhất, đúng kiểu, bỏ cột metadata.

In [39]:
# TODO: viết SQL tạo stylehub_silver.order_items: bỏ trùng theo id, ép kiểu sale_price/quantity/amount/created_at/updated_at, bỏ cột metadata
sql = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.stylehub_silver.order_items` AS
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY id
      ORDER BY updated_at DESC, _ingested_at DESC
    ) AS rn
  FROM `{GCP_PROJECT}.stylehub_bronze.order_items`
)
SELECT
  id,
  order_id,
  product_id,
  CAST(sale_price AS NUMERIC) AS sale_price,
  CAST(quantity AS INT64) AS quantity,
  CAST(amount AS NUMERIC) AS amount,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(updated_at AS TIMESTAMP) AS updated_at
FROM ranked
WHERE rn = 1
  """
bq.query(sql).result()

Kiểm tra Silver. `amount` là cột nguồn tự tính từ `sale_price * quantity`, nên số dòng lệch phải bằng 0. Số dòng bằng số id.

In [48]:
bq.query(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT id) AS n_ids, COUNTIF(amount != sale_price * quantity) AS n_mismatch FROM `{GCP_PROJECT}.stylehub_silver.order_items`").to_dataframe()

,n_rows,n_ids,n_mismatch
0,183260,183260,0


### Phần 1b: products

Nạp các snapshot của products vào Bronze, chỉ thêm.

In [27]:
# TODO: nạp products từ Raw vào Bronze
for f in glob.glob(f"{LOCAL_PATH}/products/**/*.parquet", recursive=True):
    name = "raw/stylehub/" + os.path.relpath(f, LOCAL_PATH).replace("\\", "/")
    blob = bucket.blob(name)
    if not blob.exists():
        blob.upload_from_filename(f)
# Đường dẫn Raw: raw/stylehub/products_api/date=<data_date>/products.parquet (mỗi ngày một snapshot đầy đủ)

In [28]:
job_config = bigquery.LoadJobConfig(source_format="PARQUET", write_disposition="WRITE_APPEND")
uris = [f"gs://{DEST_BUCKET}/{b.name}"
          for b in bucket.list_blobs(prefix="raw/stylehub/products/")
          if b.name.endswith(".parquet") and b.size > 10_000]  # file rỗng chỉ ~5 KB
bq.load_table_from_uri(uris, f"{GCP_PROJECT}.stylehub_bronze.products",job_config=job_config).result()

LoadJob<project=laurenprj123, location=asia-southeast1, id=3134fcef-17f8-4dcd-b280-cb8c063523c9>

Kiểm tra Bronze: số dòng lớn hơn số id nếu Raw có nhiều hơn một ngày snapshot.

In [49]:
bq.query(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT id) AS n_ids FROM `{GCP_PROJECT}.stylehub_bronze.products`").to_dataframe()

,n_rows,n_ids
0,87651,29217


Bronze → Silver: mỗi sản phẩm một dòng, giữ snapshot nạp sau cùng, ép kiểu giá.

In [42]:
# TODO: viết SQL tạo stylehub_silver.products: mỗi sản phẩm chỉ giữ snapshot mới nhất, ép kiểu cost/retail_price
sql = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.stylehub_silver.products` AS
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY id ORDER BY _ingested_at DESC) AS rn
  FROM `{GCP_PROJECT}.stylehub_bronze.products`
)
SELECT
  id,
  CAST(cost AS NUMERIC) AS cost,
  category,
  name,
  brand,
  CAST(retail_price AS NUMERIC) AS retail_price,
  department,
  sku,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(updated_at AS TIMESTAMP) AS updated_at
FROM ranked
WHERE rn = 1
"""
bq.query(sql).result()

Kiểm tra Silver: số dòng bằng số id.

In [50]:
bq.query(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT id) AS n_ids FROM `{GCP_PROJECT}.stylehub_silver.products`").to_dataframe()

,n_rows,n_ids
0,29217,29217


### Phần 2: Gold `daily_revenue`

Doanh thu theo ngày đặt đơn: số đơn và tổng `total_amount`, tính mọi đơn trừ đơn `Cancelled`. Bảng Silver `orders` đã có sẵn cột `order_date` (ngày đặt đơn).

In [ ]:
#Nạp orders_full từ Raw vào Bronze
job_config = bigquery.LoadJobConfig(source_format="PARQUET", write_disposition="WRITE_APPEND")
uris = [f"gs://{DEST_BUCKET}/{b.name}"
          for b in bucket.list_blobs(prefix="raw/stylehub/orders_full/")
          if b.name.endswith(".parquet") and b.size > 10_000]  # file rỗng chỉ ~5 KB
bq.load_table_from_uri(uris, f"{GCP_PROJECT}.stylehub_bronze.orders",job_config=job_config).result()

LoadJob<project=laurenprj123, location=asia-southeast1, id=16779c77-cec6-4283-9c5a-b813ba0ff060>

In [ ]:
sql = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.stylehub_silver.orders` AS
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY id
      ORDER BY updated_at DESC, _ingested_at DESC
    ) AS rn
  FROM `{GCP_PROJECT}.stylehub_bronze.orders`
)
SELECT
  id,
  user_id,
  CAST(total_amount AS NUMERIC) AS total_amount,
  status,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(updated_at AS TIMESTAMP) AS updated_at,
  DATE(created_at) AS order_date
FROM ranked
WHERE rn = 1
  AND status IN ('Cancelled', 'Complete', 'Processing', 'Returned', 'Shipped')
  """
bq.query(sql).result()

In [44]:
# TODO: viết SQL tạo stylehub_gold.daily_revenue từ stylehub_silver.orders
# Cột: order_date, n_orders, revenue. Doanh thu tính mọi đơn trừ đơn Cancelled. Dùng cột order_date có sẵn ở Silver.
sql = f"""
CREATE OR REPLACE TABLE `{GCP_PROJECT}.stylehub_gold.daily_revenue` AS
  SELECT
    order_date,
    COUNT(*) AS n_orders,
    SUM(total_amount) AS revenue
  FROM `{GCP_PROJECT}.stylehub_silver.orders`
  WHERE status != 'Cancelled'
  GROUP BY order_date
"""
bq.query(sql).result()

Xem 7 ngày gần nhất.

In [51]:
bq.query(f"SELECT * FROM `{GCP_PROJECT}.stylehub_gold.daily_revenue` ORDER BY order_date DESC LIMIT 7").to_dataframe()

,order_date,n_orders,revenue
0,2026-10-05,227,75327.050000000
1,2026-10-04,648,245050.210000000
2,2026-10-03,662,219166.570000000
3,2026-10-02,438,158589.200000000
4,2026-10-01,1021,350712.610000000
5,2026-09-30,1378,102042.320000000
6,2026-09-29,929,80561.400000000


Đối chiếu: tổng doanh thu trong Gold phải bằng tổng `total_amount` của các đơn không `Cancelled` trong Silver. Hai số này khác nhau nghĩa là câu SQL ở trên đang bỏ sót hoặc nhân đôi đơn.

In [52]:
bq.query(f"SELECT (SELECT SUM(revenue) FROM `{GCP_PROJECT}.stylehub_gold.daily_revenue`) AS gold, (SELECT SUM(total_amount) FROM `{GCP_PROJECT}.stylehub_silver.orders` WHERE status != 'Cancelled') AS silver").to_dataframe()

,gold,silver
0,9975372.920000000,9975372.920000000
